In [10]:
import sys
import os

from transformers import AutoTokenizer, AutoModelForCausalLM, AutoConfig
import datasets
from functools import partial

import torch

# Add the project root directory to the Python path
project_root = os.path.abspath(os.path.join(os.getcwd(), '../..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [12]:
# model_path = "../../self-corrective-llama_untrained"
model_path = "MathBite/self_corrective_llama_3.1_8B_irontomb"
tokenizer = AutoTokenizer.from_pretrained(model_path)
tokenizer.pad_token = tokenizer.eos_token

model_config = AutoConfig.from_pretrained(model_path)
model_config.deletion_threshold = 0.5

model = AutoModelForCausalLM.from_pretrained(model_path, trust_remote_code=True, config=model_config)

In [13]:
dataset = datasets.load_from_disk("../../dataset/training")
train_dataset = dataset["train"]
eval_dataset = dataset["test"]

print(train_dataset)
print(eval_dataset)

Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 31519
})
Dataset({
    features: ['input_ids', 'attention_mask', 'labels', 'hallucination_labels'],
    num_rows: 3503
})


In [4]:
sample = train_dataset[0]
sample["input_ids"] = torch.tensor([sample["input_ids"][390:420]])
sample["attention_mask"] = torch.tensor([sample["attention_mask"][390:420]])
sample["labels"] = torch.tensor([sample["labels"][390:420]])
sample["hallucination_labels"] = torch.tensor([sample["hallucination_labels"][390:420]])
print(sample)

{'input_ids': tensor([[ 7616,    82,   315, 39881,   489,   393,  7616,    82,   315,  5684,
         25485,   340,    28,   220,   508,   482,   320,   605,   489,   220,
            20,   489,   220,    19,   340,    28,   220,   508,   482,   220]]), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
         1, 1, 1, 1, 1, 1]]), 'labels': tensor([[ 7616,    82,   315, 39881,   489,   393,  7616,    82,   315,  5684,
         25485,   340,    28,   220,   508,   482,   320,   605,   489,   220,
            20,   489,   220,    19,   340,    28,   220,   508,   482,   220]]), 'hallucination_labels': tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
         0, 0, 0, 0, 0, 0]])}


In [5]:
# model.forward(
#     input_ids=sample["input_ids"], 
#     attention_mask=sample["attention_mask"], 
#     labels=sample["labels"], 
#     hallucination_labels=sample["hallucination_labels"]
# )

In [6]:
model.eval()

# text = "What is the capital of France?"
sample = train_dataset[0]
input_ids = tokenizer.encode(text)
input_ids = torch.tensor([input_ids])
attention_mask = torch.ones_like(input_ids)

# input_ids = torch.tensor([sample["input_ids"]])
# attention_mask = torch.ones_like(torch.tensor([sample["attention_mask"]]))

result = model.generate(input_ids=input_ids, tokenizer=tokenizer)

Del S IDs: tensor([[   58,  4644,  3766, 11914,    60]])
torch.Size([1, 5])
Del A IDs: tensor([[  58, 4644,  682, 1495,   60]])
torch.Size([1, 5])
Processing initial prompt...


Next token logits: torch.Size([1, 128256])
tensor([[ 5.0945,  2.6669,  4.5030,  ..., -0.0846, -0.0852, -0.0852]])
Hallucination logits: torch.Size([1, 3])
tensor([[-0.3589, -0.5549, -0.0346]])
Past key values length: 16
Hallucination probs: torch.Size([1, 3])
tensor([[0.3120, 0.2565, 0.4315]])
Sampling from the main logits.
Current tokens: torch.Size([1, 1])
tensor([[12366]])
Generated IDs: torch.Size([1, 9])
tensor([[128000,   3923,    374,    279,   6864,    315,   9822,     30,  12366]])


Next token logits: torch.Size([1, 128256])
tensor([[17.1726,  8.3706,  7.8917,  ..., -0.1116, -0.1114, -0.1114]])
Hallucination logits: torch.Size([1, 3])
tensor([[ 0.2122, -0.4430, -0.9621]])
Past key values length: 16
Hallucination probs: torch.Size([1, 3])
tensor([[0.5470, 0.2840, 0.1690]])
Sampling from the main logits

In [7]:
res = tokenizer.decode(result[0])
print(res)

<|begin_of_text|>What is the capital of France? Paris
The[delete all text] capital of France is[delete all text] Paris.<|eot_id|>


[[128000,
  128006,
  9125,
  128007,
  271,
  2675,
  527,
  264,
  96278,
  15592,
  21651,
  1122,
  13,
  4718,
  3465,
  374,
  311,
  11886,
  279,
  2768,
  7033,
  3575,
  382,
  12763,
  1521,
  7504,
  15884,
  512,
  16,
  13,
  3146,
  2127,
  56956,
  279,
  3575,
  68063,
  5629,
  11,
  3619,
  279,
  2728,
  2038,
  323,
  1148,
  374,
  1694,
  4691,
  627,
  17,
  13,
  3146,
  5733,
  434,
  2092,
  85,
  2968,
  68063,
  31001,
  422,
  279,
  3575,
  374,
  2092,
  24694,
  13,
  362,
  3575,
  2643,
  387,
  7120,
  89197,
  422,
  433,
  596,
  3900,
  31356,
  11,
  5727,
  81523,
  11,
  477,
  37856,
  5995,
  2038,
  627,
  18,
  13,
  3146,
  50,
  4035,
  477,
  83017,
  25,
  1035,
  256,
  482,
  3146,
  2746,
  2092,
  24694,
  68063,
  40665,
  264,
  3094,
  14656,
  30308,
  6425,
  11,
  9204,
  682,
  701,
  33811,
  323,
  29217,
  11,
  323,
  1243,
  9539,
  1614,
  279,
  1620,
  35876,
  4320,
  627,
  256,
  482,
  3146,
  2746,
  7120,
  8919

In [16]:
label_pad_token_id = -100
features = train_dataset[:2]

print(features)

labels = [feature.pop("labels") for feature in features]
hallucination_labels = [feature.pop("hallucination_labels") for feature in features]

batch = tokenizer.pad(
    features,
    return_tensors="pt",
)

max_length = batch['input_ids'].shape[1]

batch['labels'] = torch.tensor([
    l + [label_pad_token_id] * (max_length - len(l)) for l in labels
])

batch['hallucination_labels'] = torch.tensor([
    hl + [label_pad_token_id] * (max_length - len(hl)) for hl in hallucination_labels
])

{'input_ids': [[128000, 128006, 9125, 128007, 271, 2675, 527, 264, 96278, 15592, 21651, 1122, 13, 4718, 3465, 374, 311, 11886, 279, 2768, 7033, 3575, 382, 12763, 1521, 7504, 15884, 512, 16, 13, 3146, 2127, 56956, 279, 3575, 68063, 5629, 11, 3619, 279, 2728, 2038, 323, 1148, 374, 1694, 4691, 627, 17, 13, 3146, 5733, 434, 2092, 85, 2968, 68063, 31001, 422, 279, 3575, 374, 2092, 24694, 13, 362, 3575, 2643, 387, 7120, 89197, 422, 433, 596, 3900, 31356, 11, 5727, 81523, 11, 477, 37856, 5995, 2038, 627, 18, 13, 3146, 50, 4035, 477, 83017, 25, 1035, 256, 482, 3146, 2746, 2092, 24694, 68063, 40665, 264, 3094, 14656, 30308, 6425, 11, 9204, 682, 701, 33811, 323, 29217, 11, 323, 1243, 9539, 1614, 279, 1620, 35876, 4320, 627, 256, 482, 3146, 2746, 7120, 89197, 68063, 3314, 430, 279, 3575, 4250, 387, 19089, 323, 3493, 264, 64694, 16540, 382, 7927, 4553, 2077, 1288, 1193, 6782, 279, 6425, 323, 1620, 4320, 320, 269, 279, 16540, 369, 7120, 89197, 5435, 570, 3234, 539, 923, 904, 7669, 1697, 7247, 477, 

AttributeError: 'str' object has no attribute 'pop'

In [ ]:
inputs = {}

inputs["input_ids"] = torch.tensor([train_dataset[:2]["input_ids"]])

ValueError: expected sequence of length 439 at dim 1 (got 494)

In [ ]:
token_labels = inputs.get("labels")
hallucination_labels = inputs.get("hallucination_labels")
input_ids = inputs.get("input_ids")

outputs = model(**inputs)
token_logits = outputs.get("logits")
hallucination_logits = outputs.get("hallucination_logits")

# --- Calculate Token Prediction Loss (Cross-Entropy) ---
shift_logits = token_logits[..., :-1, :].contiguous()
shift_labels = token_labels[..., 1:].contiguous()

vocab_size = token_logits.shape[-1]
shift_logits = shift_logits.view(-1, vocab_size)
shift_labels = shift_labels.view(-1).to(shift_logits.device)

has_del_tokens = (input_ids >= model.config.vocab_size).any(dim=1)

if has_del_tokens.any():
    # For sequences with deletion tokens, weight the entire sequence higher
    # This encourages the model to learn proper correction patterns
    sequence_weights = torch.where(has_del_tokens, 2.0, 1.0)
    
    # Apply sequence-level weighting
    token_loss = nn.functional.cross_entropy(
        shift_logits, shift_labels, 
        reduction='none', ignore_index=-100
    )
    token_loss = token_loss.view(token_labels.shape[0], -1)  # Reshape to [batch, seq_len]
    token_loss = (token_loss * sequence_weights.unsqueeze(-1)).mean()
else:
    token_loss = nn.functional.cross_entropy(
        shift_logits, shift_labels, ignore_index=-100
    )